# 4. Ensembles de Arboles de Decision (paralelo con mclapply)


Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente local


Esta parte se debe correr con un kernel de R local.
<br>En Jupyter, seleccionar el kernel **R** antes de ejecutar el notebook.


Los archivos persistentes quedan en el repo local: datasets en `datasets/` y resultados en `exp/`.


In [16]:
# Seteo local: resuelve el root del repo (funciona desde raiz, src/ensembles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")


Para correr localmente, el dataset debe estar en `datasets/` dentro del repo.

<br>Si se va a subir a Kaggle, copiar `kaggle.json` a la raiz del repo antes de correr la siguiente celda. La celda lo instala en `~/.kaggle/kaggle.json` con permisos correctos.


In [17]:
dir.create(DATA_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)

dataset_local <- file.path(DATA_DIR, "dataset_pequeno.csv")
if (!file.exists(dataset_local)) {
  stop("No encuentro el dataset en: ", dataset_local)
}

if (file.exists(KAGGLE_JSON)) {
  kaggle_dir <- path.expand("~/.kaggle")
  dir.create(kaggle_dir, recursive = TRUE, showWarnings = FALSE)
  file.copy(KAGGLE_JSON, file.path(kaggle_dir, "kaggle.json"), overwrite = TRUE)
  Sys.chmod(file.path(kaggle_dir, "kaggle.json"), mode = "0600")
} else {
  message("No encontre kaggle.json en la raiz del repo. Solo es necesario para hacer submit a Kaggle.")
}




---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el kernel de **R**.


limpio el ambiente de R

In [18]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 08 10:22:24 2026"

In [19]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,811512,43.4,1489770,79.6,NA,1489770,79.6
Vcells,1506685,11.5,87095487,664.5,49152,90268748,688.7


In [20]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")
# mismo paquete de locks que el grid search (z290)
if (!require("flock", quietly = TRUE)) {
  install.packages("flock")
  require("flock")
}
# evita oversubscription OpenMP/BLAS dentro de cada fork
data.table::setDTthreads(1L)


Aqui debe cargar SU semilla primigenia

In [21]:
PARAM <- list()
PARAM$semilla_primigenia <- 300089

# parametros  arbol
# entreno cada arbol con solo 50% de las variables variables
#  por ahora, es fijo
PARAM$feature_fraction <- 0.5

PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 500
PARAM$rpart$minbucket <- 125
PARAM$rpart$maxdepth <- 8

# voy a generar 512 arboles,
#  a mas arboles mas tiempo de proceso y MEJOR MODELO,
#  pero ganancias marginales
PARAM$num_trees_max <- 512

# no subir a Kaggle hasta que digamos lo contrario
PARAM$submit_kaggle <- FALSE


In [22]:
# carpeta de trabajo (re-resuelve paths por si hubo rm() arriba)
# Seteo local: resuelve el root del repo (funciona desde raiz, src/ensembles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento <- "exp4020_parallel"
dir.create(file.path(EXP_DIR, experimento), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento))


In [23]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"))


In [24]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [25]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [26]:
# que tamanos de ensemble grabo a disco
grabar <- c(1, 2, 4, 8, 16, 32, 64, 128, 256, 384, 512)

In [27]:
# tb_probs y tb_prediccion se arman despues del mclapply
# (probs por arbol + acumulados ordenados 1..N)


In [28]:
set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

In [29]:
PARAM$num_trees_max

[1] 512

In [30]:
# Version paralela:
# 1) Pre-genero features (misma semilla => arboles equivalentes al for-loop)
# 2) mclapply entrena todos los arboles y arma la tabla completa
# 3) Acumulados en orden 1..N
# SIN upload a Kaggle en esta celda
# Progreso: flock::lock sobre progreso_arboles.txt (igual que z290 grid search)

set.seed(PARAM$semilla_primigenia)

qty_campos_a_utilizar <- as.integer(length(campos_buenos) * PARAM$feature_fraction)

campos_por_arbol <- lapply(seq(PARAM$num_trees_max), function(i) {
  sample(campos_buenos, qty_campos_a_utilizar)
})

# --- progreso thread-safe entre workers (paquete flock, como en z290) ---
log_file <- "progreso_arboles.txt"
err_file <- "progreso_arboles.errors.txt"
counter_file <- "progreso_arboles.count"
writeLines(character(0), log_file)
writeLines(character(0), err_file)
writeLines("0", counter_file)

FormatTime <- function(ts) format(ts, "%Y-%m-%d %H:%M:%S")

log_progress <- function(arbolito, ok = TRUE, err_msg = NULL) {
  # lock del archivo de log (API de flock: lock / unlock)
  lk <- lock(log_file)
  on.exit(unlock(lk), add = TRUE)

  n_done <- as.integer(readLines(counter_file, n = 1L, warn = FALSE))
  if (is.na(n_done)) n_done <- 0L
  n_done <- n_done + 1L
  writeLines(as.character(n_done), counter_file)

  cat(
    sprintf(
      "[%s] arbol %03d %s | completados %d/%d\n",
      FormatTime(Sys.time()),
      arbolito,
      if (ok) "listo" else "ERROR",
      n_done,
      PARAM$num_trees_max
    ),
    file = log_file,
    append = TRUE
  )

  if (!ok && !is.null(err_msg)) {
    cat(
      sprintf("[%s] arbol %03d: %s\n", FormatTime(Sys.time()), arbolito, err_msg),
      file = err_file,
      append = TRUE
    )
  }

  invisible(n_done)
}

entrenar_arbolito <- function(arbolito) {
  data.table::setDTthreads(1L)

  out <- tryCatch(
    {
      campos_random <- campos_por_arbol[[arbolito]]
      formulita <- paste0("clase_ternaria ~ ", paste(campos_random, collapse = " + "))

      modelo <- rpart(
        formulita,
        data = dtrain,
        xval = 0,
        control = PARAM$rpart
      )

      prediccion <- predict(modelo, dfuture, type = "prob")
      if (!("BAJA+2" %in% colnames(prediccion))) {
        stop(
          "predict no tiene columna BAJA+2; cols=",
          paste(colnames(prediccion), collapse = ",")
        )
      }
      as.numeric(prediccion[, "BAJA+2"])
    },
    error = function(e) e
  )

  if (inherits(out, "error")) {
    log_progress(arbolito, ok = FALSE, err_msg = conditionMessage(out))
    stop(conditionMessage(out))
  }

  log_progress(arbolito, ok = TRUE)
  out
}

ncores <- max(1L, min(8L, parallel::detectCores() - 2L))
message(
  "Entrenando ", PARAM$num_trees_max, " arboles con mclapply en ", ncores, " cores\n",
  "Progreso: ", normalizePath(log_file, mustWork = FALSE), "\n",
  "  tail -f ", log_file, "\n",
  "Errores (si hay): ", err_file
)

probs_list <- mclapply(
  X = seq(PARAM$num_trees_max),
  FUN = entrenar_arbolito,
  mc.cores = ncores,
  mc.preschedule = TRUE,
  mc.silent = FALSE
)

bad <- vapply(probs_list, inherits, logical(1), what = "try-error")
n_ok <- sum(!bad)
n_bad <- sum(bad)
message(
  "Entrenamiento terminado: ", n_ok, "/", PARAM$num_trees_max, " arboles OK",
  if (n_bad > 0) paste0(" | ", n_bad, " fallaron (ver ", err_file, ")") else ""
)

if (n_bad > 0) {
  message("Primer error de worker:\n", as.character(probs_list[[which(bad)[1]]]))
  stop("mclapply fallo en ", n_bad, " arboles. Revisar ", err_file)
}

mat_probs <- do.call(cbind, probs_list)
storage.mode(mat_probs) <- "numeric"
colnames(mat_probs) <- sprintf("arbol_%03d", seq(PARAM$num_trees_max))

tb_probs <- data.table(
  numero_de_cliente = dfuture$numero_de_cliente,
  mat_probs
)

mat_acum <- t(apply(mat_probs, 1L, cumsum))
colnames(mat_acum) <- sprintf("acum_%03d", seq(PARAM$num_trees_max))

tb_prediccion <- data.table(
  numero_de_cliente = dfuture$numero_de_cliente,
  mat_acum
)

fwrite(tb_probs, file = "tb_probs.csv")
fwrite(tb_prediccion, file = "tb_prediccion_acum.csv")

message(
  "Listo. tb_probs / tb_prediccion en memoria y en disco.\n",
  "Siguiente: celda de export CSVs. Upload Kaggle en celda aparte (off por defecto)."
)


Entrenando 512 arboles con mclapply en 8 cores
Progreso: /Users/selewaut/Code Projects/master/dm2026b/exp/exp4020_parallel/progreso_arboles.txt
  tail -f progreso_arboles.txt
Errores (si hay): progreso_arboles.errors.txt

Entrenamiento terminado: 512/512 arboles OK

Listo. tb_probs / tb_prediccion en memoria y en disco.
Siguiente: celda de export CSVs. Upload Kaggle en celda aparte (off por defecto).



#### Export CSVs de submission (sin Kaggle)
Corre cuando `tb_prediccion` ya exista (despues del mclapply).


In [31]:
# Genera CSVs de submission para cada tamano en `grabar` (sin subir a Kaggle)
# Requiere tb_prediccion en memoria o tb_prediccion_acum.csv en el cwd del experimento

if (!exists("tb_prediccion") || is.null(tb_prediccion)) {
  if (file.exists("tb_prediccion_acum.csv")) {
    tb_prediccion <- fread("tb_prediccion_acum.csv")
    message("Cargue tb_prediccion desde tb_prediccion_acum.csv")
  } else {
    stop("No hay tb_prediccion en memoria ni tb_prediccion_acum.csv en el cwd")
  }
}

for (arbolito in grabar) {
  message("Escribiendo ensemble de ", arbolito, " arboles")
  col_acum <- sprintf("acum_%03d", arbolito)
  umbral_corte <- (1 / 40) * arbolito

  tb_out <- tb_prediccion[, .(
    numero_de_cliente,
    Predicted = as.numeric(get(col_acum) > umbral_corte)
  )]

  archivo_kaggle <- paste0("KA420_", sprintf("%.3d", arbolito), ".csv")
  fwrite(tb_out, file = archivo_kaggle, sep = ",")
  message("  -> ", archivo_kaggle, " (umbral ", umbral_corte, ")")
}

message("CSVs listos. Upload: celda siguiente (PARAM$submit_kaggle <- TRUE).")


Escribiendo ensemble de 1 arboles

  -> KA420_001.csv (umbral 0.025)

Escribiendo ensemble de 2 arboles

  -> KA420_002.csv (umbral 0.05)

Escribiendo ensemble de 4 arboles

  -> KA420_004.csv (umbral 0.1)

Escribiendo ensemble de 8 arboles

  -> KA420_008.csv (umbral 0.2)

Escribiendo ensemble de 16 arboles

  -> KA420_016.csv (umbral 0.4)

Escribiendo ensemble de 32 arboles

  -> KA420_032.csv (umbral 0.8)

Escribiendo ensemble de 64 arboles

  -> KA420_064.csv (umbral 1.6)

Escribiendo ensemble de 128 arboles

  -> KA420_128.csv (umbral 3.2)

Escribiendo ensemble de 256 arboles

  -> KA420_256.csv (umbral 6.4)

Escribiendo ensemble de 384 arboles

  -> KA420_384.csv (umbral 9.6)

Escribiendo ensemble de 512 arboles

  -> KA420_512.csv (umbral 12.8)

CSVs listos. Upload: celda siguiente (PARAM$submit_kaggle <- TRUE).



#### Upload a Kaggle (deshabilitado por defecto)
Solo cuando quieras subir: `PARAM$submit_kaggle <- TRUE` y correr la celda.


In [36]:
PARAM$submit_kaggle <- TRUE

In [37]:
# Submit a Kaggle — solo despues de tener la tabla y los KA420_*.csv
# Deshabilitado por defecto.

if (is.null(PARAM$submit_kaggle)) {
  PARAM$submit_kaggle <- FALSE
}

if (!isTRUE(PARAM$submit_kaggle)) {
  message(
    "Upload deshabilitado (PARAM$submit_kaggle = FALSE).\n",
    "Para habilitar: PARAM$submit_kaggle <- TRUE y re-correr esta celda."
  )
} else {
  for (arbolito in grabar) {
    archivo_kaggle <- paste0("KA420_", sprintf("%.3d", arbolito), ".csv")
    if (!file.exists(archivo_kaggle)) {
      warning("No existe ", archivo_kaggle, " — salteo")
      next
    }

    message("Subiendo ", archivo_kaggle)
    comando <- "kaggle competitions submit"
    competencia <- "-c data-mining-inicial-2026-b"
    arch <- paste("-f", archivo_kaggle)
    mensaje <- paste0(
      "-m 'cp=", PARAM$rpart$cp,
      "  minsplit=", PARAM$rpart$minsplit,
      "  minbucket=", PARAM$rpart$minbucket,
      " maxdepth=", PARAM$rpart$maxdepth,
      " parallel'"
    )
    linea <- paste(comando, competencia, arch, mensaje)
    salida <- system(linea, intern = TRUE)
    cat(salida, sep = "\n")
    cat("\n")
  }
  message("Submissions finalizadas.")
}


Subiendo KA420_001.csv



82 submissions remaining today.
Successfully submitted to Data Mining, Inicial 2026 B



Subiendo KA420_002.csv



81 submissions remaining today.
Successfully submitted to Data Mining, Inicial 2026 B



Subiendo KA420_004.csv



...

Subiendo KA420_384.csv



73 submissions remaining today.
Successfully submitted to Data Mining, Inicial 2026 B



Subiendo KA420_512.csv



72 submissions remaining today.
Successfully submitted to Data Mining, Inicial 2026 B



Submissions finalizadas.



In [33]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 08 11:05:19 2026"



---

